In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
import streamlit as st
import datetime as dt

# ------------------------------------
# data load

In [ ]:
amazon_df=pd.read_csv('Amazon Sale Report.csv')
pd.set_option('display.max_columns',None)
pd.set_option('display.expand_frame_repr',False)
amazon_df.columns=amazon_df.columns.str.replace('-',' ').str.lower().str.strip()

# ------------------------------------
# data understanding

In [ ]:
amazon_df.head()
#target column = status

for i in amazon_df.columns:
    print(f'unique values in {i}:{amazon_df[i].unique()}')
    print('-'*20)
#for loop on all columns in this dataframe to see all unique values in it

amazon_numerical=['qty','amount']

amazon_categorical=['status','fulfilment','ship service level','style',
                    'category','ship city','ship state',
                    'b2b','promotion ids','month','day','month periods','promotion ids update']


amazon_df.info()
# NOTE (date column to datetime data type)
#date column have a different data type(object) i wanna make it (datetime)
#amount,shipcity,ship state,ship postal code,promotion ids(have a missing values)

amazon_df.describe()
#average order quantity = 0.90
#average order price = 648.5

amazon_df.shape
#(128975, 24)

# explaination of understanding
#### 1- i made two empty lists to separate between categorical columns and numerical columns
#### 2- i looked at the columns to understand the meaning of each column(.head())
#### 3- i made a for loop to see all unique values for each column
#### 4- i saw the data info to see the missing values and data types for each column
#### 5- i saw if i can make a column features to improve my analysis

# -------------------------------------
# data cleaning

In [ ]:
clean_df=amazon_df.drop(columns=['index','sku','courier status','currency','ship country','fulfilled by','unnamed: 22'])
#before dropping these columns i saw the unique values and i didn't find any feature to improve my analysis
#so i made a new dataframe contains only interesting columns to start work on it


In [ ]:

clean_df['date']=pd.to_datetime(clean_df['date'])
clean_df['date'].dtype
clean_df.info()
#########################################################
clean_df[clean_df['amount']==0]
clean_df[clean_df['qty']==0].groupby('status')['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#i checked if i have a wrong values in amount column and i found it
#i checked if i have a wrong values in quantity column and i found it 
condition=(clean_df['amount']==0)
clean_df.loc[condition,'amount'] = np.nan
condition2=(clean_df['qty']==0) & (clean_df['status']!='Cancelled')
clean_df.loc[condition2,'qty']= np.nan
#########################################################
clean_df.duplicated().sum()
clean_df.duplicated(keep=False)
#to check duplicates i got 6 duplicated rows
#########################################################
#outliers function to get the outliers percentage for numerical columns
######################################################### 
def outliers_fun(x):
    x=x.dropna()
    Q1=x.quantile(0.25)
    Q3=x.quantile(0.75)
    IQR=Q3-Q1
    upper_limit=Q3+1.5*IQR
    lower_limit=Q1-1.5*IQR
    outliers=x[(x>upper_limit) | (x<lower_limit)]
    return round(len(outliers)/len(x)*100,2)
#########################################################
fill=['amount']
report={
    i:outliers_fun(clean_df[i])
    for i in fill
}
report
#amount report=3.15(filling with mean)
#and another columns = categorical (filling with mode)
##########################################################
#filling the missing values and dropping duplicates
clean_df['amount']=clean_df['amount'].fillna(clean_df['amount'].mean())
clean_df['ship city']=clean_df['ship city'].fillna(clean_df['ship city'].mode()[0])
clean_df['ship state']=clean_df['ship state'].fillna(clean_df['ship state'].mode()[0])
clean_df['ship postal code']=clean_df['ship postal code'].fillna(clean_df['ship postal code'].mode()[0])
clean_df['promotion ids']=clean_df['promotion ids'].fillna(clean_df['promotion ids'].mode()[0])
clean_df['qty']=clean_df['qty'].fillna(clean_df['qty'].mode()[0])
clean_df=clean_df.drop_duplicates()
clean_df.info()
clean_df.head()

# explaination of cleaning
#### 1- i made a new dataframe only contains the columns that i need it in my analysis
#### 2- i changed the data type for date column
#### 3- i checked if i have wrong values in columns=amount & qty ,and i found some of wrong values
#### 4- i made a condition if this condition=true,the values that brought this condition will be transformed into nan
#### 5- i checked the duplicates and i found 6 of duplicated rows
#### 6- i made a outliers function to check the percentage of outliers for numerical columns,>7% fill with mean, above fill with median
#### 7- filling the missing values depending on the type of column, categorical with mode,numerical according to their outliers percentage
#### 8- i dropped all duplicates

# extract some of column features

In [ ]:
clean_df['month']=clean_df['date'].dt.month_name()
clean_df['day']=clean_df['date'].dt.day_name()
####################################################
def fun(x):
    if x.day in range(21,32):
        return 'q3'
    elif x.day in range(11,21):
        return 'q2'
    elif x.day in range(1,11):
        return 'q1'
    return 'other'

clean_df['month periods']=clean_df['date'].apply(fun)
######################################################
clean_df['promotion ids update']=clean_df['promotion ids'].str.split('AAT').str[0].str.strip()
clean_df['promotion ids update'].unique()
clean_df['promotion ids update']=clean_df['promotion ids update'].str.split('2015').str[0].str.strip()
clean_df['promotion ids update'].unique()
clean_df['promotion ids update']=clean_df['promotion ids update'].str.split('A12R').str[0].str.strip()
clean_df['promotion ids update'].unique()
clean_df['promotion ids update']=clean_df['promotion ids update'].str.split('AY').str[0].str.strip()
clean_df['promotion ids update'].unique()
clean_df['promotion ids update']=clean_df['promotion ids update'].str.split('-').str[0].str.strip()
clean_df['promotion ids update'].unique()

def clean_promo_logic(text):
    if 'IN Core Free Shipping' in text:
        return 'in core free shipping'
    elif 'Amazon PLCC Free' in text:
        return 'amazon plcc free financing universal merchant'
    elif 'Duplicated' in text :
        return 'duplicated'
    else:
        return 'vpc coupon'
clean_df['promotion ids update']=clean_df['promotion ids'].apply(clean_promo_logic)
clean_df['promotion ids update'].value_counts()
#########################################################
clean_df.head()
clean_df.info()
clean_df.to_csv('cleaned amazon dataframe')

# explaination to column features
#### 1- i made a column = month name and column = day name to make a datetime analysis clearer
#### 2- i made a column = month period to check what is the sales in each month period?
#### 3- i hade difficualty dealing with promotion ids column,therefore, i created a new column containing only the values i need it,and discarded all the values that hinder my analysis
#### i made it with basic python method and funcion making to check if the condition = true ,return the values i gave to you
#### (before that i checked all unique values for this column carefully to make sure that these are only the values i need)

#### 4-i made a last check whether the data was ready for analysis or not,then i saved my clean dataframe to csv to start the next step

# -----------------------------------------------
# exploratory data analysis(EDA)

# 1- i wanna see how much of orders in each category

In [ ]:
clean_df.groupby('category')['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#the most orders placed(set=50284,kurta=49877)
#then(western dress=15500,top=10622)
#then(ethnic dress=1159,blouse=926,bottom=440,saree=164,dupatta=3)

# plot 1

In [ ]:
fig1=px.histogram(clean_df,x='category')
fig1.show()

# 2- i wanna see how much of these orders was received by the buyer or what does there status?

In [ ]:
clean_df[clean_df['status']=='Shipped'].groupby(['status','category'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#shipped core(kurta=30794)(set=30665)(western dress=7480)(top=7143)(ethnic dress=755)(blouse=623)(bottom=222)(saree=1199)(dupatta=3)
#----------------------------------------------
clean_df[clean_df['status']=='Cancelled'].groupby(['status','category'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#cancelled core(kurta=7337)(set=7255)(western dress=2122)(top=1276)(ethnic dress=145)(blouse=116)(bottom=60)(saree=21)(dupatta= zero)
#----------------------------------------------
clean_df[clean_df['status']=='Shipped - Delivered to Buyer'].groupby(['status','category'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#Shipped-Delivered to Buyer(set=10644)(kurta=10451)(western dress=5190)(top=1920)(ethnic dress=234)(blouse=169)(bottom=129)(saree=22)(dupatta=zero)
#----------------------------------------------
clean_df[clean_df['status']=='Shipped - Returned to Seller'].groupby(['status','category'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#Shipped-Returned to Seller(Set=766)(kurta=715)(western dress=314)(top=124)(ethnic dress=16)(blouse=12)(bottom=5)(saree=1)(dupatta=zero)
#----------------------------------------------
clean_df[clean_df['status']=='Shipped - Rejected by Buyer'].groupby(['status','category'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#Shipped-Rejected by Buyer(set=6)(top=2)(kurta=2)(western dress=1) all another=zero
#----------------------------------------------
clean_df[clean_df['status']=='Shipped - Lost in Transit'].groupby(['status','category'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#Shipped-Lost in Transit(set=2)(kurta=2)(western dress=1) all another=zero
#----------------------------------------------
clean_df[clean_df['status']=='Shipped - Out for Delivery'].groupby(['status','category'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#Shipped-Out for Delivery(set=19)(western dress=6)(top=5)(kurta=5) all another=zero
#----------------------------------------------
clean_df[clean_df['status']=='Shipped - Returning to Seller'].groupby(['status','category'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#Shipped-Returning to Seller(set=73)(kurta=35)(western dress=28)(top=8)(bottom=1)
#----------------------------------------------
clean_df[clean_df['status']=='Shipped - Picked Up'].groupby(['status','category'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#Shipped-Picked Up(set=406)(kurta=297)(western dress=186)(top=69)(bottom=10)(blouse=2)(ethnic dress=2)(saree=1)all another=zero
#----------------------------------------------
clean_df[clean_df['status']=='Shipped - Damaged'].groupby(['status','category'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#Shipped - Damaged only one order=western dress
#----------------------------------------------
clean_df[clean_df['status']=='Pending'].groupby(['status','category'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#Pending(set=252)(kurta=248)(western dress=92)(top=55)(ethnic dress=7)(blouse=3)(bottom=1)all another=zero
#----------------------------------------------
clean_df[clean_df['status']=='Pending - Waiting for Pick Up'].groupby(['status','category'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#Pending-Waiting for Pick Up(set=108)(western dress=79)(kurta=72)(top=19)(bottom=2)(blouse=1) all another =zero
#----------------------------------------------
clean_df[clean_df['status']=='Shipping'].groupby(['status','category'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#Shipping(set=6)(top1)(kurta=1)

# plot 2

In [ ]:
fig2=px.histogram(clean_df,x='status',color='category')
fig2.show()

In [ ]:
fig3=px.pie(clean_df,names='status')
fig3.show()

# 3- i wanna see how much of orders in each ship service level and what does there status?

In [ ]:
clean_df.groupby(['status','ship service level'])['amount'].sum().reset_index().sort_values(by='status',ascending=True)
clean_df[clean_df['ship service level']=='Expedited'].groupby(['status','ship service level'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#expedited shipping core = {Shipped : 76779 ,
#cancelled : 11423 ,
#pending : 1 }
clean_df[clean_df['ship service level']=='Standard'].groupby(['status','ship service level'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#standard shipping core={Shipped - Delivered to Buyer : 28769,
#Cancelled : 6909,
#Shipped - Returned to Seller : 1953
#Shipped : 1025
#Shipped - Picked Up : 973
#Pending - Waiting for Pick Up : 281
#Pending : 245
#Shipped - Returning to Seller : 145
#Shipped - Out for Delivery : 35
#Shipped - Rejected by Buyer	: 11
#Shipping : 8
#Shipped - Lost in Transit : 5
#Shipped - Damaged : 1
#}
clean_df.groupby(['fulfilment','ship service level'])['order id'].count().reset_index().sort_values(by='fulfilment',ascending=False)
#NOTE the expedited core have a highest orders placed but haven't any order received (big red flag)

# plot 3-

In [ ]:
fig4=px.histogram(clean_df,x='ship service level',color='status')
fig4.show()

# 4- i wanna see how much of orders in type b2b and for which category and what does there status?

In [ ]:
clean_df[clean_df['b2b']==True].groupby(['category','b2b','status'])
[['amount','order id','qty']].agg({'order id':'count','amount':'sum','qty':'sum'}
                                        ).reset_index().sort_values(by='status',ascending=False)
#NOTE i wanna make a campagin for b2b customers focused on weastern dress,kurta,sett,top and make for them a exclusive offers,and faster shipping core

# plot 4

In [ ]:
fig5=px.histogram(clean_df,x='b2b')
fig5.show()

In [ ]:
fig6=px.histogram(clean_df,x='category',color='status',facet_col='b2b')
fig6.show()

# 5- i wanna see which one of offers (promotion ids) had a status = delivered to buyer and which one had a cancelled or another status 
#relations between target and and promotion ids agg=sum(amount),count(order id)

In [ ]:
clean_df[clean_df['status']=='Cancelled'].groupby(['status','promotion ids update']
                                    ).agg({'order id':'count','amount':'sum'}
                                    ).reset_index().sort_values(by='amount',ascending=False)
#clean_df['promotion ids'].unique()

In [ ]:
clean_df[clean_df['status']=='Pending'].groupby(['status','promotion ids update']
                                    ).agg({'amount':'sum','order id':'count'}
                                    ).reset_index().sort_values(by='amount',ascending=False).head(35)


In [ ]:
clean_df[clean_df['status']=='Pending - Waiting for Pick Up'].groupby(['status','promotion ids update']
                                            ).agg({'amount':'sum','order id':'count'}
                                            ).reset_index().sort_values(by='amount',ascending=False).head(35)


In [ ]:
clean_df[clean_df['status']=='Shipped'].groupby(['status','promotion ids update']
                                            ).agg({'amount':'sum','order id':'count'}
                                            ).reset_index().sort_values(by='amount',ascending=False).head(35)


In [ ]:
clean_df[clean_df['status']=='Shipped - Damaged'].groupby(['status','promotion ids update']
                                            ).agg({'amount':'sum','order id':'count'}
                                            ).reset_index().sort_values(by='amount',ascending=False).head(35)


In [ ]:
clean_df[clean_df['status']=='Shipped - Delivered to Buyer'].groupby(['status','promotion ids update']
                                                        ).agg({'amount':'sum','order id':'count'}
                                                        ).reset_index().sort_values(by='amount',ascending=False).head(35)


In [ ]:
clean_df[clean_df['status']=='Shipped - Lost in Transit'].groupby(['status','promotion ids update']
                                                        ).agg({'amount':'sum','order id':'count'}
                                                        ).reset_index().sort_values(by='amount',ascending=False).head(35)


In [ ]:
clean_df[clean_df['status']=='Shipped - Out for Delivery'].groupby(['status','promotion ids update']
                                                            ).agg({'amount':'sum','order id':'count'}
                                                            ).reset_index().sort_values(by='amount',ascending=False).head(35)


In [ ]:
clean_df[clean_df['status']=='Shipped - Picked Up'].groupby(['status','promotion ids update']
                                                    ).agg({'amount':'sum','order id':'count'}
                                                    ).reset_index().sort_values(by='amount',ascending=False).head(35)


In [ ]:
clean_df[clean_df['status']=='Shipped - Rejected by Buyer'].groupby(['status','promotion ids update']
                                                                ).agg({'amount':'sum','order id':'count'}
                                                                ).reset_index().sort_values(by='amount',ascending=False).head(35)


In [ ]:
clean_df[clean_df['status']=='Shipped - Returning to Seller'].groupby(['status','promotion ids update']
                                                                ).agg({'amount':'sum','order id':'count'}
                                                                ).reset_index().sort_values(by='amount',ascending=False)

# plot 5

In [ ]:
fig7=px.histogram(clean_df,x='promotion ids update',color='status')
fig7.show()
# i wanna make a column feature to make each promotion like that, in core free shipping

# 6- i wanna see which city or state have a most qty and amount in there orders and what is there status
#relations between target and(ship city,ship state).agg(qty)

In [ ]:
clean_df.groupby(['ship city','ship state']).agg({'order id':'count','qty':'sum','amount':'sum'}
                                                ).reset_index().sort_values(by='order id',ascending=False)
#top 5 cities and states have orders:
#BENGALURU	KARNATAKA
#HYDERABAD	TELANGANA
#MUMBAI	   MAHARASHTRA
#NEW DELHI	      DELHI	
#CHENNAI      TAMIL NADU'''

In [ ]:
clean_df['status'].unique()
clean_df.groupby(['ship city','ship state']).agg({'order id':'count','qty':'sum','amount':'sum'}
            ).reset_index().sort_values(by='amount',ascending=False).head(10)
#top 5 citys and states in core recieved by buyer
'''BENGALURU	KARNATAKA
    HYDERABAD	TELANGANA
    MUMBAI	   MAHARASHTRA
  NEW DELHI	      DELHI	
   CHENNAI      TAMIL NADU'''

In [ ]:
clean_df[clean_df['status']=='Cancelled'].groupby(
    ['status','ship city','ship state']).agg({'order id':'count','qty':'sum','amount':'sum'}
           ).reset_index().sort_values(by='qty',ascending=False).head(20)


In [ ]:
clean_df[clean_df['status']=='Shipped - Delivered to Buyer'].groupby(
    ['status','ship city','ship state']).agg({'order id':'count','qty':'sum','amount':'sum'},
            ).reset_index().sort_values(by='qty',ascending=False).head(20)

In [ ]:
clean_df[clean_df['status']=='Shipped'].groupby(
    ['status','ship city','ship state']).agg({'order id':'count','qty':'sum','amount':'sum'}
            ).reset_index().sort_values(by='qty',ascending=False).head(20)

In [ ]:
clean_df[clean_df['status']=='Shipped - Returned to Seller'].groupby(
    ['status','ship city','ship state']).agg({'order id':'count','qty':'sum','amount':'sum'}
        ).reset_index().sort_values(by='qty',ascending=False).head(20)

In [ ]:
clean_df[clean_df['status']=='Shipped - Rejected by Buyer'].groupby(
    ['status','ship city','ship state']).agg({'order id':'count','qty':'sum','amount':'sum'}
            ).reset_index().sort_values(by='qty',ascending=False).head(20)

In [ ]:
clean_df[clean_df['status']=='Shipped - Lost in Transit'].groupby(
    ['status','ship city','ship state']).agg({'order id':'count','qty':'sum','amount':'sum'}
            ).reset_index().sort_values(by='qty',ascending=False).head(20)

In [ ]:
clean_df[clean_df['status']=='Shipped - Out for Delivery'].groupby(
                ['status','ship city','ship state']).agg({'order id':'count','qty':'sum','amount':'sum'}
                ).reset_index().sort_values(by='qty',ascending=False).head(20)

In [ ]:
clean_df[clean_df['status']=='Shipped - Returning to Seller'].groupby(
    ['status','ship city','ship state']).agg({'order id':'count','qty':'sum','amount':'sum'}
            ).reset_index().sort_values(by='qty',ascending=False).head(20)

In [ ]:
clean_df[clean_df['status']=='Shipped - Picked Up'].groupby(
    ['status','ship city','ship state']).agg({'order id':'count','qty':'sum','amount':'sum'}
            ).reset_index().sort_values(by='qty',ascending=False).head(20)

In [ ]:
clean_df[clean_df['status']=='Pending'].groupby(
    ['status','ship city','ship state']).agg({'order id':'count','qty':'sum','amount':'sum'}
            ).reset_index().sort_values(by='qty',ascending=False).head(20)

In [ ]:
clean_df[clean_df['status']=='Pending - Waiting for Pick Up'].groupby(
    ['status','ship city','ship state']).agg({'order id':'count','qty':'sum','amount':'sum'}
            ).reset_index().sort_values(by='qty',ascending=False).head(20)

In [ ]:
clean_df[clean_df['status']=='Shipped - Damaged'].groupby(
    ['status','ship city','ship state']).agg({'order id':'count','qty':'sum','amount':'sum'}
            ).reset_index().sort_values(by='qty',ascending=False).head(20)

In [ ]:
clean_df[clean_df['status']=='Shipping'].groupby(
    ['status','ship city','ship state']).agg({'order id':'count','qty':'sum','amount':'sum'}
            ).reset_index().sort_values(by='qty',ascending=False).head(20)

# plot 6

In [ ]:
state_orders=clean_df.groupby('ship state').agg({'order id':'count','qty':'sum','amount':'sum'}).reset_index()
fig10=px.bar(state_orders.sort_values(by='order id',ascending=False).head(10),x='ship state',y='order id')
fig10.show()


In [ ]:
city_orders=clean_df.groupby('ship city').agg({'order id':'count','qty':'sum','amount':'sum'}).reset_index()
fig11=px.bar(city_orders.sort_values(by='order id',ascending=False).head(10),x='ship city',y='order id')
fig11.show()

In [ ]:
city_orders=clean_df.groupby(['ship city','status']).agg({'order id':'count','qty':'sum','amount':'sum'}).reset_index()
fig11=px.bar(city_orders.sort_values(by='order id',ascending=False).head(10),x='ship city',y='order id',facet_col='status')
fig11.show()

# 7- i wanna see which month and which day have a highest orders number

In [ ]:
print(clean_df.groupby('month').agg({'order id':'count'}).reset_index().sort_values(by='order id',ascending=False))
#highest monthes=(april,may)then(june),sales in march is soo weak
clean_df.groupby('day').agg({'order id':'count'}).reset_index().sort_values(by='order id',ascending=False)
#sales in all days was too close but top 3 is sunday,tuesday,wednesday

# plot 7

In [ ]:
fig10=px.histogram(clean_df,x='month')
fig10.show()

In [ ]:
fig11=px.histogram(clean_df,x='day')
fig11.show()

# 8- relations between fulfilment and target

In [ ]:
clean_df[clean_df['status']=='Shipped - Delivered to Buyer'].groupby('fulfilment')['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#all orders have status == delivered to buyer fulfilmented by merchant not amazon and this=28769
#cancelled =(amazon=11468)(merchant=6861)
#returned to seller =(merchant=1953)
clean_df.groupby('fulfilment')['order id'].count().reset_index().sort_values(by='order id',ascending=False)
#89692 for amazon , 39277 for merchant
clean_df.groupby(['status','fulfilment'])['order id'].count().reset_index().sort_values(by='fulfilment',ascending=False)


# plot 8

In [ ]:
fig12=px.histogram(clean_df,x='fulfilment',color='status')
fig12.show()

# 9- i wanna see which size had a highest order numbers and from which category

In [ ]:
clean_df.groupby(['size','category'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)

# plot 9

In [ ]:
fig13=px.histogram(clean_df,x='size',color='category')
fig13.show()

In [ ]:
fig14=px.histogram(clean_df,x='size')
fig14.show()

# 10- i wanna check which month period have a highest orders numbers

In [ ]:
clean_df.groupby('month periods')['order id'].count().reset_index().sort_values(by='order id',ascending=False)

# plot 10

In [ ]:
fig15=px.histogram(clean_df,x='month periods')
fig15.show()

# explaination for analysis
1-i checked each category with their order counts

2-i check for order counts with their category for each order status to see which category have a highest orders delivered to buyer and which one have another status to see if i had a problem in any product category

3-i checked for all ship service levels to see how much of orders in each one and what does their status

4-i checked for how much of orders in b2b type and what does the highest product catgeory they order and what does their orders status

5-i checked about what kind of promotions have a status==delivered to buyer and what kind in another status

6-i checked for which city and which state have a most quantity and amount in their orders and what does their status

7-i checked about which month and which day have a highest orders number

8- i checked about which fulfilment type had a highest orders number and what does their status

9-i checked about which size had a higher orders number and for which category

10-i checked if the orders be higher or lower in any quarter of the month


# analysis insights and recommendations
1- The most orders placed in set and kurta categories then western dress and top then….
The most orders had a cancellation status in kurta and set then western dress and top then….
The most orders received by buyer in set and kurta then western dress and top then……
The most orders have shipped status was kurta and set then western dress and top
(so i wanna focused in this categories in my restocking and campaigns)
the most product had a order delivery success percentage = (western dress with 33.4%)


2- Expedited ship level didn’t have any order= received by buyer,only have shipped=76779
Cancelled=11423
Standard ship level have a 28769 orders received by buyer and cancelled=6909
And returned to seller=1953,shipped=1025
And all another status was in standard level
(So i have a problem in the expedited ship service level and i wanna improve it to achieve a higher orders delivered to buyer or i wanna change the shipping method) 
all orders in expedited ship level was fulfiled by amazon!
and almost all orders in standard ship level was fulfiled by merchant


3- The highest orders in b2b type was kurta and set,western dress,top
(So i wanna make a campaign to b2b type focused on this products with ship service level=expedited (really expedited))



4- The highest orders was received by buyer was had a promotion=free financing 
I wanna make this offer again by making a contracts with value,aman,or any related company,
The highest orders have a cancellation status was in core free shipping because the shipping company is so bad i wanna make it again after changing the shipping method 
And i wanna make a vpc coupon and check it capabilities after changing the shipping company


5- Top 5 city’s was received orders and make orders is(bengaluru,hyderabad,mumbai,new delhi,chennai,pune)
Topstates
(karnataka,telangana,maharashtra,delhi,tamilnadu,uttar pradesh)
#maharashtra state have a lot of orders 
(i wanna give it more attention in my campaigns
And if i have any city or state have a high cancellation percentage in their orders i will make a system to take a deposit,or you can only paid by credit card from any client lived in this destinations)

6- I want to increase the advertising budget for April, May, and June. i also want to see if we’re not advertising in March. If we’re not, we’ll try make ads in March. If we are making ads but not generating revenue, we’ll try to avoid March and reduce the budget significantly.
All days of the week was had similar order volumes,but i can focused on sunday


7- All orders have status== delivered to buyer fulfilmented by merchant not amazon
Cancelled =(amazon=11468)(merchant=6861)
Returned to seller=(merchant1953)
Orders for amazon=89692  merchant=39277
So i wanna ship my orders by myself or other shipping company


8- I wanna focus on restocking sizes = (xs,s,m,l,xl,xxl,xxxl)
And reduce quantity from another sizes


9- All month periods was had similar order volumes.,but i can focused on q1 cause they had a most orders number and some of employees taking their salary at this period

# -------------------
# deployment

In [ ]:
%%writefile amazonAPP.py
import pandas as pd
import streamlit as st
import plotly.express as px
st.set_page_config(page_title='amazon results',layout='wide')
df=pd.read_csv('cleaned amazon dataframe')
st.title('amazon orders analysis dashboard')
st.sidebar.header('filters')


page=st.sidebar.radio('choose section',
                      ['overview',
                       'dashboard',
                       'orders by categories',
                       'order status',
                       'shipping service',
                       'b2b analysis',
                       'promotions',
                       'geography',
                       'time analysis',
                       'fulfilment vs status',
                       'size analysis',
                       'month period analysis'
                      ])


select_status=st.sidebar.multiselect('select statu',df['status'].unique())
select_day=st.sidebar.multiselect('select day',df['day'].unique())
select_month=st.sidebar.multiselect('select month',df['month'].unique())
select_category=st.sidebar.multiselect('select category',df['category'].unique())
select_size=st.sidebar.multiselect('select size',df['size'].unique())
select_b2b_or_b2c=st.sidebar.radio('select b2b or not',df['b2b'].unique())
select_type_of_fulfilment=st.sidebar.radio('select fulfilment',df['fulfilment'].unique())
select_state=st.sidebar.multiselect('select state',df['ship state'].unique())
select_city=st.sidebar.multiselect('select city',df['ship city'].unique())
select_period=st.sidebar.selectbox('select period',df['month periods'].unique())
select_promotion=st.sidebar.multiselect('select promotion',df['promotion ids update'].unique())
min_qty,max_qty=st.slider('select quantity',int(df['qty'].min()),int(df['qty'].max()),
                          (int(df['qty'].min()),int(df['qty'].max())))
min_amount,max_amount=st.slider('select amount',float(df['amount'].min()),float(df['amount'].max()),
                                (float(df['amount'].min()),float(df['amount'].max())))
select_ship_service_level=st.sidebar.radio('select ship service level',df['ship service level'].unique())


filtered_df=df.copy()
if select_status:
    filtered_df=filtered_df[filtered_df['status'].isin(select_status)]
if select_day:
    filtered_df=filtered_df[filtered_df['day'].isin(select_day)]
if select_month:
    filtered_df=filtered_df[filtered_df['month'].isin(select_month)]
if select_size:
    filtered_df=filtered_df[filtered_df['size'].isin(select_size)]
if select_b2b_or_b2c:
    filtered_df=filtered_df[filtered_df['b2b']==select_b2b_or_b2c]
if select_state:
    filtered_df=filtered_df[filtered_df['ship state'].isin(select_state)]
if select_city:
    filtered_df=filtered_df[filtered_df['ship city'].isin(select_city)]
if select_period:
    filtered_df=filtered_df[filtered_df['month periods']==select_period]
if select_promotion:
    filtered_df=filtered_df[filtered_df['promotion ids update'].isin(select_promotion)]
if select_type_of_fulfilment:
    filtered_df=filtered_df[filtered_df['fulfilment']==select_type_of_fulfilment]
if select_ship_service_level:
    filtered_df=filtered_df[filtered_df['ship service level']==select_ship_service_level]
if select_category:
    filtered_df=filtered_df[filtered_df['category']==select_category]
    

min_value,max_value=min_qty,max_qty
filtered_df=filtered_df[(filtered_df['qty']>=min_value)&(filtered_df['qty']<=max_value)]
min_value2,max_value2=min_amount,max_amount
filtered_df=filtered_df[(filtered_df['amount']>=min_value2)&(filtered_df['amount']<=max_value2)]


st.dataframe(filtered_df)

#dashboard

#over view:
if page == 'overview':
    st.header('dataset overview')
    col1,col2,col3=st.columns(3)
    col1.metric('total orders = ',len(filtered_df))
    col2.metric('total cities = ',filtered_df['ship city'].nunique())
    col3.metric('total of categories = ',filtered_df['category'].nunique())
#1 orders by category:
elif page=='orders by categories':
    st.header('orders by category')
    fig1=px.histogram(filtered_df,x='category')
    st.plotly_chart(fig1,use_container_width=True)
#2 orders status
elif page=='order status':
    st.header('order status distribution')
    fig2=px.histogram(filtered_df,x='status')
    st.plotly_chart(fig2,use_container_width=True)

#3 ship service level
elif page=='shipping service':
    st.header('shipping service level vs status')
    fig3=px.histogram(filtered_df,x='ship service level',color='status')
    st.plotly_chart(fig3,use_container_width=True)

#4 b2b analysis
elif page=='b2b analysis':
    st.header('b2b orders by category and status')
    fig4=px.histogram(filtered_df,x='category',color='b2b')
    st.plotly_chart(fig4,use_container_width=True)
    fig5=px.histogram(filtered_df,x='status',color='b2b')
    st.plotly_chart(fig5,use_container_width=True)
    
#5 promotions

elif page=='promotions':
    st.header('promotion ids vs status')
    fig6=px.histogram(filtered_df,x='promotion ids',color='status')
    st.plotly_chart(fig6,use_container_width=True)
    
#6 geography analysis
elif page=='geography':
    st.header('orders by cities and states')
    state_orders=filtered_df.groupby('ship state').size().reset_index(name='state orders')
    cities_orders=filtered_df.groupby('ship city').size().reset_index(name='city orders')
    fig7=px.bar(state_orders.sort_values(by='state orders',ascending=False).head(10),x='ship state',y='state orders')
    st.plotly_chart(fig7,use_container_width=True)
    fig8=px.bar(cities_orders.sort_values(by='city orders',ascending=False).head(10),x='ship city',y='city orders')
    st.plotly_chart(fig8,use_container_width=True)

#7 time analysis
elif page=='time analysis':
    st.header('orders over time')
    fig9=px.histogram(filtered_df,x='month')
    st.plotly_chart(fig9,use_container_width=True)
    fig10=px.histogram(filtered_df,x='day')
    st.plotly_chart(fig10,use_container_width=True)
    
#8 fulfilment vs status
elif page== 'fulfilment vs status':
    st.header('fulfilment vs status')
    fig11=px.histogram(filtered_df,x='fulfilment',color='status')
    st.plotly_chart(fig11,use_container_width=True)
    
#9 size analysis
elif page=='size analysis':
    st.header('size vs category')
    fig12=px.histogram(filtered_df,x='size',color='category')
    st.plotly_chart(fig12,use_container_width=True)
#month periods analysis
elif page== 'month period analysis':
    st.header('orders between month periods')
    fig13=px.histogram(filtered_df,x='month periods')
    st.plotly_chart(fig13,use_container_width=True)

In [ ]:
! streamlit run amazonAPP.py

# expalination for deployment

1-i made a select section by sidebar and radio,because if the stakeholder wanna see each insight plot visually he can make this

2-i made a seletion filters for each column in the dataframe because if he wanna check for anything in the his data he can make this

3-i made a filtered dataframe then make a simple relation system to be valid and interactive with my filters

4-